In [106]:
# Code to map Maven elements to Origami HAI elements
# Using semantic similarity library
# Author: Sandeep Chintabathina
# April 2026

In [107]:
# For each Maven data element, check if there is an exact or close matching data element in the origami HAI package
# The output will list the maven element alongside the origami element and other supporting attributes

In [108]:
from sentence_transformers import SentenceTransformer


In [109]:
import pandas as pd
import numpy as np

In [110]:
# Read maven elements
df_maven = pd.read_csv('output/maven_all_with_mmg_and_cste.csv',na_filter=False)
len(df_maven)

9501

In [111]:
df_maven.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 9501 entries, 0 to 9500
Data columns (total 9 columns):
 #   Column                 Non-Null Count  Dtype 
---  ------                 --------------  ----- 
 0   Model                  9501 non-null   object
 1   question_number        9501 non-null   int64 
 2   depth                  9501 non-null   int64 
 3   Question               9501 non-null   object
 4   Question.Package.ID    9501 non-null   object
 5   Question.Package.Name  9501 non-null   object
 6   Message                9501 non-null   object
 7   mmg_element            9501 non-null   object
 8   cste_element           9501 non-null   object
dtypes: int64(2), object(7)
memory usage: 668.2+ KB


In [112]:
def clean_up(name):
    new_name =''
    for ch in name:
        # If space, digits, uppercase letters, lowercase letters, or underscore, add it to new name otherwise ignore
        if (ord(ch)==32 or 48<=ord(ch)<=57 or 65<=ord(ch)<=90 or 97<=ord(ch)<=122 or ord(ch)==95 ):
            new_name+=ch
    return new_name.strip()

In [113]:
clean_up('[sdfs] 34_AS ^#~')

'sdfs 34_AS'

In [114]:
# Clean up questions or data elements
df_maven.loc[:,'Question'] =[ clean_up(q) for q in df_maven.loc[:,'Question']]

In [115]:
df_maven.loc[:,'Model'].value_counts(dropna=False)

Model
General                  3866
STD                      2425
Coinfection               923
Hepatitis                 901
Childhood Lead            606
Cluster                   425
Lead investigation        255
STD EHR and screening      85
Aggregate                  15
Name: count, dtype: int64

In [116]:
# duplicates
df_maven[df_maven.duplicated(['Question'],keep=False)]

,Model,question_number,depth,Question,Question.Package.ID,Question.Package.Name,Message,mmg_element,cste_element
0,General,1,0,DISEASE,ADMINISTRATIVE,Administrative,Disease,,
2,General,3,0,ALL_BT,ADMINISTRATIVE,Administrative,Show for all BT diseases,,
4,General,5,0,ALL_GEN,ADMINISTRATIVE,Administrative,Show for all general diseases,,
5,General,6,0,ALL_FLU,ADMINISTRATIVE,Administrative,Show for all FLU diseases,,
6,General,7,0,ALL_ENT,ADMINISTRATIVE,Administrative,All Enteric diseases,,
...,...,...,...,...,...,...,...,...,...
9420,STD,2345,0,PRODUCT_DESCRIPTION,,,,,
9421,STD,2346,0,ISO_QUAR_CASE_INSTRUCTIONS,,,,,
9422,STD,2347,0,ISO_QUAR_CONTACT_INSTRUCTIONS,,,,,
9438,STD,2363,0,UNAFFILIATED_CASES,,,,,


In [117]:
# Keeping the first appearance of question
df_maven = df_maven.drop_duplicates(['Question'])

In [118]:
df_maven[df_maven.duplicated(['Question'],keep=False)]

,Model,question_number,depth,Question,Question.Package.ID,Question.Package.Name,Message,mmg_element,cste_element


In [119]:
df_maven = df_maven.reset_index(drop=True)

In [120]:
df_maven

,Model,question_number,depth,Question,Question.Package.ID,Question.Package.Name,Message,mmg_element,cste_element
0,General,1,0,DISEASE,ADMINISTRATIVE,Administrative,Disease,,
1,General,2,0,ALL_ARBO,ADMINISTRATIVE,Administrative,All Arboviruses,,
2,General,3,0,ALL_BT,ADMINISTRATIVE,Administrative,Show for all BT diseases,,
3,General,4,0,ALL_COVID,ADMINISTRATIVE,Administrative,All COVID disease codes,,
4,General,5,0,ALL_GEN,ADMINISTRATIVE,Administrative,Show for all general diseases,,
...,...,...,...,...,...,...,...,...,...
7801,STD,2421,0,DATE_SENT_CDC,,,,,
7802,STD,2422,0,SURRG_CONTACT_FLAG,,,,,
7803,STD,2423,0,SURRG_PARTNER_FLAG,,,,,
7804,STD,2424,0,SURRG_FIRST_GEN_FLAG,,,,,


In [121]:
# HAI elements
df_hai = pd.read_csv('origami/csv/Custom__CaseHAI.csv',na_filter=False)
len(df_hai)

757

In [122]:
df_hai.loc[:,'Label'].value_counts(dropna=False)

Label
                                            228
HC_OUTSIDE_US_COUNTRY                         2
AFIM_RR_colon_screen                          2
DATE_OF_LAST_DAY_IN_SCHOOL_WORK_FACILITY      2
INTERNATIONAL_DESTINATION_OTHER               2
                                           ... 
CDI_22b2                                      1
CDI_22b1                                      1
CDI_22b                                       1
CDI_22a2                                      1
AFIM_RR_inter_oth                             1
Name: count, Length: 519, dtype: int64

In [123]:
# Get the two columns of interest and non-blank values
df_hai = df_hai[['Field','Label']]
df_hai = df_hai[df_hai['Label']!='']
len(df_hai)

529

In [124]:
df_hai = df_hai.reset_index(drop=True)

In [125]:
df_hai

,Field,Label
0,CaseHAIID,Case HAIID
1,ClientID,Client
2,EntryUserID,Entry User
3,EntryDate,Entry Date
4,ModifiedUserID,Modified User
...,...,...
524,CustomText130,CDI_21m_oth
525,CustomText131,CDI_36e1_oth
526,CustomText132,colonized_cdiff_facility
527,CustomText133,CDI_27_other


In [126]:
# Original label to be used later for output
df_hai.rename(columns={'Label':'original_label'},inplace=True)
df_hai.columns

Index(['Field', 'original_label'], dtype='object')

In [127]:
# Remove only the first occurrence of [] in a string
def remove_squares(somestring):
    if ']' in somestring:
        for i in range(len(somestring)):
            if somestring[i]==']':
                return somestring[i+1:].strip()
    return somestring.strip()


In [128]:
remove_squares('asd[asdasd]asc[obas]da')

'asc[obas]da'

In [129]:
# Process the column values (eliminate the values in square brackets)
df_hai.loc[:,'Label'] = [ remove_squares(label) for label in df_hai.loc[:,'original_label']]

In [130]:
# Clean up column names in both data sets
df_hai.loc[:,'Label'] =[ clean_up(q) for q in df_hai.loc[:,'Label']]

In [131]:
df_hai.loc[:,'Label'].value_counts()

Label
HC_OUTSIDE_US_COUNTRY                       2
AFIM_RR_ed_aware                            2
DATE_OF_LAST_DAY_IN_SCHOOL_WORK_FACILITY    2
Other type of test                          2
Test Method                                 2
                                           ..
CDI_22b                                     1
CDI_22a2                                    1
CDI_22a1                                    1
CDI_22a                                     1
AFIM_RR_inter_oth                           1
Name: count, Length: 515, dtype: int64

In [132]:
# duplicates labels do not exist 
df_hai[df_hai.duplicated(['Label'],keep=False)]

,Field,original_label,Label
108,CustomDate4,DATE_OF_LAST_DAY_IN_SCHOOL_WORK_FACILITY,DATE_OF_LAST_DAY_IN_SCHOOL_WORK_FACILITY
123,CustomDate19,DATE_OF_LAST_DAY_IN_SCHOOL_WORK_FACILITY,DATE_OF_LAST_DAY_IN_SCHOOL_WORK_FACILITY
135,CustomMultiSelectCode6,HC_OUTSIDE_US_COUNTRY,HC_OUTSIDE_US_COUNTRY
141,CustomMultiSelectCode12,AFIM_RR_inter,AFIM_RR_inter
142,CustomMultiSelectCode13,AFIM_RR_colon_screen,AFIM_RR_colon_screen
143,CustomMultiSelectCode14,AFIM_RR_ed_aware,AFIM_RR_ed_aware
144,CustomMultiSelectCode15,AFIM_RR_environ,AFIM_RR_environ
145,CustomMultiSelectCode16,AFIM_RR_surveillance,AFIM_RR_surveillance
146,CustomMultiSelectCode17,AFIM_RR_presentation,AFIM_RR_presentation
184,CustomText4,[OBX 5.9]Other type of test,Other type of test


In [133]:
# dropping them
df_hai = df_hai.drop_duplicates(['Label'])

In [134]:
df_hai[df_hai.duplicated(['Label'],keep=False)]

,Field,original_label,Label


In [135]:
df_hai = df_hai.reset_index(drop=True)
df_hai

,Field,original_label,Label
0,CaseHAIID,Case HAIID,Case HAIID
1,ClientID,Client,Client
2,EntryUserID,Entry User,Entry User
3,EntryDate,Entry Date,Entry Date
4,ModifiedUserID,Modified User,Modified User
...,...,...,...
510,CustomDate74,AFIM_RR_IPCA_DATE,AFIM_RR_IPCA_DATE
511,CustomText130,CDI_21m_oth,CDI_21m_oth
512,CustomText132,colonized_cdiff_facility,colonized_cdiff_facility
513,CustomText133,CDI_27_other,CDI_27_other


In [136]:
# Do a semantic match between Maven elements and Origami HAI labels

In [137]:
maven_elements = [str.replace(maven_de,'_',' ') for maven_de in df_maven.Question]
origami_elements = [str.replace(col,'_',' ') for col in df_hai.Label]

In [138]:
#origami_elements

In [139]:
model = SentenceTransformer("all-MiniLM-L6-v2")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [140]:
# Compute embeddings
embeddings_maven = model.encode(maven_elements)
embeddings_origami = model.encode(origami_elements)

In [141]:
print(len(embeddings_origami))
print(len(embeddings_maven))
embeddings_origami
embeddings_maven

515
7806


array([[-0.04004316,  0.03481774, -0.02303074, ...,  0.06413301,
         0.07289726,  0.05008497],
       [-0.03188282, -0.06779679, -0.13589898, ..., -0.08035976,
        -0.11267023,  0.05088616],
       [-0.04829377, -0.06161015, -0.00392639, ..., -0.04496994,
        -0.02579335, -0.03141985],
       ...,
       [ 0.0012861 ,  0.02914467, -0.06241952, ..., -0.08015   ,
        -0.02389724,  0.05160841],
       [-0.036089  ,  0.06744839, -0.03735485, ..., -0.023555  ,
         0.01138324,  0.04567831],
       [-0.09624534,  0.06870063,  0.06528897, ..., -0.05273273,
         0.00868748,  0.0212906 ]], dtype=float32)

In [142]:
# Compute cosine similarities
similarities = model.similarity(embeddings_maven,embeddings_origami)

In [143]:
similarities

tensor([[ 0.2804,  0.2485,  0.1516,  ...,  0.0132,  0.1643,  0.1619],
        [ 0.1929,  0.1684,  0.1530,  ..., -0.0545,  0.1732,  0.1816],
        [ 0.1880,  0.1656,  0.0895,  ..., -0.0193,  0.2432,  0.1787],
        ...,
        [ 0.0693,  0.0991,  0.0643,  ...,  0.0550,  0.1108,  0.2346],
        [ 0.1005, -0.0727,  0.0555,  ...,  0.0759,  0.1301,  0.2291],
        [ 0.2505,  0.0416,  0.0596,  ...,  0.1967,  0.1390,  0.2639]])

In [144]:
print(type(similarities))
len(similarities)

<class 'torch.Tensor'>


7806

In [145]:
df_similarities = pd.DataFrame(similarities,index=df_maven.Question,columns=df_hai.original_label)

In [146]:
df_similarities.to_csv('output/similarity_scores_HAI.csv')

In [147]:
df_maven = df_maven.reset_index(drop=True)
df_maven.columns

Index(['Model', 'question_number', 'depth', 'Question', 'Question.Package.ID',
       'Question.Package.Name', 'Message', 'mmg_element', 'cste_element'],
      dtype='object')

In [148]:
df_output = df_maven[['Model','depth','Question','Message','mmg_element','cste_element']]

In [95]:
#df_similarities.info()

In [149]:
#For every maven element (index), identify origami elements (column) that is the best match

filtered=[]
for idx in df_similarities.index:
    # Looking for columns with score higher than 0.75
    v = df_similarities.loc[idx,:]>0.75

    # Get the column with the maximum score
    max=0.0
    max_col=''
    for i,col in enumerate(df_similarities.columns):
        if v.iloc[i]==True:
            #print('value',df_similarities.loc[idx,col])
            if (df_similarities.loc[idx,col] > max):
                max = df_similarities.loc[idx,col]
                max_col=col

    #filtered.append({col:df_similarities.loc[idx,col]})
    #print(idx+','+max_col+':'+str(max)+'\n')
    filtered.append({'Question':idx,'target_label':max_col})



In [150]:
df_filtered = pd.DataFrame(filtered)
print(len(df_filtered))
#df_filtered.loc[:,'target_label'].value_counts(dropna=False)

7806


In [151]:
df_merge = pd.merge(df_filtered,df_hai,left_on='target_label',right_on='original_label',how='left')

In [152]:
df_merge

,Question,target_label,Field,original_label,Label
0,DISEASE,,NaN,NaN,NaN
1,ALL_ARBO,,NaN,NaN,NaN
2,ALL_BT,,NaN,NaN,NaN
3,ALL_COVID,,NaN,NaN,NaN
4,ALL_GEN,,NaN,NaN,NaN
...,...,...,...,...,...
7801,DATE_SENT_CDC,DATE_SENT_TO_CDC,CustomDate20,DATE_SENT_TO_CDC,DATE_SENT_TO_CDC
7802,SURRG_CONTACT_FLAG,,NaN,NaN,NaN
7803,SURRG_PARTNER_FLAG,,NaN,NaN,NaN
7804,SURRG_FIRST_GEN_FLAG,,NaN,NaN,NaN


In [153]:
# merge the above merged df with output dataframe
df_final = pd.merge(df_output,df_merge,on='Question')
#df_final

In [154]:
df_final.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7806 entries, 0 to 7805
Data columns (total 10 columns):
 #   Column          Non-Null Count  Dtype 
---  ------          --------------  ----- 
 0   Model           7806 non-null   object
 1   depth           7806 non-null   int64 
 2   Question        7806 non-null   object
 3   Message         7806 non-null   object
 4   mmg_element     7806 non-null   object
 5   cste_element    7806 non-null   object
 6   target_label    7806 non-null   object
 7   Field           166 non-null    object
 8   original_label  166 non-null    object
 9   Label           166 non-null    object
dtypes: int64(1), object(9)
memory usage: 610.0+ KB


In [155]:
# Add, rename, remove columns as needed
df_final['target_table'] = ['' if f=='' else 'Custom__CaseHAI' for f in df_final.target_label]
df_final=df_final.rename(columns={'Field':'target_field'})
df_final=df_final.drop('Label',axis=1)


In [156]:
df_final = df_final[["Model",'Question','target_table','target_field','target_label','mmg_element','cste_element','Message','depth']]
#df_final

In [157]:
df_final

,Model,Question,target_table,target_field,target_label,mmg_element,cste_element,Message,depth
0,General,DISEASE,,NaN,,,,Disease,0
1,General,ALL_ARBO,,NaN,,,,All Arboviruses,0
2,General,ALL_BT,,NaN,,,,Show for all BT diseases,0
3,General,ALL_COVID,,NaN,,,,All COVID disease codes,0
4,General,ALL_GEN,,NaN,,,,Show for all general diseases,0
...,...,...,...,...,...,...,...,...,...
7801,STD,DATE_SENT_CDC,Custom__CaseHAI,CustomDate20,DATE_SENT_TO_CDC,,,,0
7802,STD,SURRG_CONTACT_FLAG,,NaN,,,,,0
7803,STD,SURRG_PARTNER_FLAG,,NaN,,,,,0
7804,STD,SURRG_FIRST_GEN_FLAG,,NaN,,,,,0


In [516]:
#df_final[df_final['target_field'].isna()]

In [158]:
df_final.to_csv('output/maven_HAI_origami_similarity_match.csv',index=False)